In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
import os
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import warnings
warnings.filterwarnings('ignore')

data_dir = "../data"
train_df = pd.read_csv(os.path.join(data_dir, 'processed_train.csv'), index_col='Date', parse_dates=True)
test_df = pd.read_csv(os.path.join(data_dir, 'processed_test.csv'), index_col='Date', parse_dates=True)

X_train_t = torch.tensor(train_df[['GOLD_TL', 'VIX', 'BOND_2Y']].values, dtype=torch.float32)
y_train_t = torch.tensor(train_df['BIST100'].values, dtype=torch.float32).view(-1, 1)
X_test_t = torch.tensor(test_df[['GOLD_TL', 'VIX', 'BOND_2Y']].values, dtype=torch.float32)
y_test = test_df['BIST100'].values

# --- MODELLER ---
class Gaussian_ANFIS(nn.Module):
    def __init__(self, num_inputs, num_rules):
        super().__init__()
        self.num_rules, self.num_inputs = num_rules, num_inputs
        self.mu = nn.Parameter(torch.rand(num_rules, num_inputs))
        self.sigma = nn.Parameter(torch.rand(num_rules, num_inputs) + 0.1) 
        self.cw = nn.Parameter(torch.rand(num_rules, num_inputs))
        self.cb = nn.Parameter(torch.rand(num_rules, 1))

    def forward(self, x):
        bs = x.shape[0]
        x_exp = x.unsqueeze(1).expand(-1, self.num_rules, -1)
        mu_exp = self.mu.unsqueeze(0).expand(bs, -1, -1)
        sig_exp = self.sigma.unsqueeze(0).expand(bs, -1, -1)
        
        fuzzy_vals = torch.exp(-0.5 * ((x_exp - mu_exp) / (sig_exp.abs() + 1e-5))**2)
        firing = torch.prod(fuzzy_vals, dim=2)
        norm_firing = firing / (torch.sum(firing, dim=1, keepdim=True) + 1e-8)
        
        c_out = torch.matmul(x, self.cw.T) + self.cb.T
        return torch.sum(norm_firing * c_out, dim=1, keepdim=True)

class GBell_ANFIS(nn.Module):
    def __init__(self, num_inputs, num_rules):
        super().__init__()
        self.num_rules, self.num_inputs = num_rules, num_inputs
        self.a = nn.Parameter(torch.ones(num_rules, num_inputs) * 0.3)
        self.b = nn.Parameter(torch.ones(num_rules, num_inputs) * 2.0)
        self.c = nn.Parameter(torch.linspace(0, 1, num_rules).unsqueeze(1).repeat(1, num_inputs))
        self.cw = nn.Parameter(torch.rand(num_rules, num_inputs))
        self.cb = nn.Parameter(torch.rand(num_rules, 1))

    def forward(self, x):
        bs = x.shape[0]
        x_exp = x.unsqueeze(1).expand(-1, self.num_rules, -1)
        a_exp = self.a.unsqueeze(0).expand(bs, -1, -1)
        b_exp = self.b.unsqueeze(0).expand(bs, -1, -1)
        c_exp = self.c.unsqueeze(0).expand(bs, -1, -1)
        
        # Matematiksel Sınırlar (Eğitimin patlamasını önler)
        a_safe = a_exp.abs() + 1e-3
        b_safe = F.softplus(b_exp) + 0.5 
        
        fuzzy_vals = 1.0 / (1.0 + torch.pow(torch.abs((x_exp - c_exp) / a_safe), 2 * b_safe))
        firing = torch.prod(fuzzy_vals, dim=2)
        norm_firing = firing / (torch.sum(firing, dim=1, keepdim=True) + 1e-8)
        
        c_out = torch.matmul(x, self.cw.T) + self.cb.T
        return torch.sum(norm_firing * c_out, dim=1, keepdim=True)

# --- ABLATION DENEY KURULUMU ---
configs = {
    'A0': {'model': Gaussian_ANFIS, 'rules': 8, 'lr': 0.01, 'ep': 1500, 'name': 'Baseline (Gauss, 8K)'},
    'A1': {'model': GBell_ANFIS,    'rules': 8, 'lr': 0.01, 'ep': 1500, 'name': 'Sadece G-Bell'},
    'A2': {'model': Gaussian_ANFIS, 'rules': 16,'lr': 0.01, 'ep': 1500, 'name': 'Sadece 16 Kural'},
    'A3': {'model': Gaussian_ANFIS, 'rules': 8, 'lr': 0.005,'ep': 2500, 'name': 'Sadece Eğitim (lr/ep)'},
    'A4': {'model': GBell_ANFIS,    'rules': 16,'lr': 0.005,'ep': 2500, 'name': 'Proposed (GBell, 16K, Eğitim)'}
}

results = []
print("Ablation Deneyleri Başlıyor (Lütfen bekleyiniz, bu işlem biraz sürebilir)...\n")

for key, cfg in configs.items():
    mse_list, mae_list, r2_list = [], [], []
    print(f"Test Ediliyor -> {cfg['name']} ({key})")
    
    # 5 Farklı Rastgelelik (Seed) ile Güvenilirlik Testi
    for seed in range(5):
        torch.manual_seed(seed)
        np.random.seed(seed)
        
        model = cfg['model'](num_inputs=3, num_rules=cfg['rules'])
        optimizer = torch.optim.Adam(model.parameters(), lr=cfg['lr'])
        criterion = nn.MSELoss()
        
        # Eğitim Döngüsü
        model.train()
        for epoch in range(cfg['ep']):
            optimizer.zero_grad()
            loss = criterion(model(X_train_t), y_train_t)
            loss.backward()
            optimizer.step()
            
        # Test Aşaması
        model.eval()
        with torch.no_grad():
            preds = model(X_test_t).numpy().flatten()
            
        mse_list.append(mean_squared_error(y_test, preds))
        mae_list.append(mean_absolute_error(y_test, preds))
        r2_list.append(r2_score(y_test, preds))
        
    # Ortalamaları ve Standart Sapmaları Kaydet
    results.append({
        'Model (Varyant)': cfg['name'],
        'MSE': f"{np.mean(mse_list):.5f} ± {np.std(mse_list):.5f}",
        'MAE': f"{np.mean(mae_list):.5f} ± {np.std(mae_list):.5f}",
        'R2': f"{np.mean(r2_list):.5f} ± {np.std(r2_list):.5f}"
    })

# Sonuçları Tablo Olarak Bastır
results_df = pd.DataFrame(results)
print("\n" + "="*70)
print("DENEY SONUÇLARI (5 Seed Ortalaması)")
print("="*70)
print(results_df.to_string(index=False))

Ablation Deneyleri Başlıyor (Lütfen bekleyiniz, bu işlem biraz sürebilir)...

Test Ediliyor -> Baseline (Gauss, 8K) (A0)
Test Ediliyor -> Sadece G-Bell (A1)
Test Ediliyor -> Sadece 16 Kural (A2)
Test Ediliyor -> Sadece Eğitim (lr/ep) (A3)
Test Ediliyor -> Proposed (GBell, 16K, Eğitim) (A4)

DENEY SONUÇLARI (5 Seed Ortalaması)
              Model (Varyant)               MSE               MAE                 R2
         Baseline (Gauss, 8K) 0.04919 ± 0.00401 0.15680 ± 0.00470 -0.49509 ± 0.12177
                Sadece G-Bell 0.05260 ± 0.00867 0.16118 ± 0.00828 -0.59871 ± 0.26354
              Sadece 16 Kural 0.05460 ± 0.00809 0.16083 ± 0.00795 -0.65953 ± 0.24584
        Sadece Eğitim (lr/ep) 0.04728 ± 0.00364 0.15698 ± 0.00303 -0.43692 ± 0.11053
Proposed (GBell, 16K, Eğitim) 0.06787 ± 0.01489 0.17628 ± 0.01302 -1.06298 ± 0.45244
